In [4]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm

from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor


In [5]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [6]:
covariates = ["dolvol_lag2", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"]

In [ ]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=180):
    """
    Génère des splits temporels selon la logique décrite :
    - Train cumulatif (augmente d'un an à chaque refit)
    - Train 8 ans 
    - Validation = fenêtre fixe glissante de 2 an 
    - Test = 1 an 
    - Avance de step_months à chaque itération : 12 mois

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop si on n'a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Avancer d'un step (ex : 12 mois) pour le prochain refit
        start += step_months

    return splits

In [16]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [17]:
#Mesures : 
#R²
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    """
    Calcule le success ratio = proportion de signes correctement prédits.
    y_true et y_pred doivent être des array-like de même longueur.
    ignore_zero : si True, on ignore les observations où y_true == 0.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  # au cas où toutes les valeurs sont nulles

    return (sign_true == sign_pred).mean()


In [18]:
#Permet de récupérer x et y 
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target]) #garde toutes les colonnes mais enlève excess return
    y = subset[target]
    return x, y

In [ ]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)



100%|██████████| 10/10 [01:48<00:00, 10.84s/it]


In [38]:
"""
XGB : XGBoost Regressor
Hyperparamètres :
- n_estimators : nombre d’arbres
- max_depth : profondeur max
- eta : learning rate
"""

param_grid_xgb = {
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01],
    'n_estimators': [300],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.5, 1, 2]
}

# Pour calculer les R² globaux
y_trainval_xgb = []

# Stocke les R² par split
r2_in_split_xgb = []
r2_oos_split_xgb = []

y_pred_xgb = []

selected_splits = [3, 5, 7, 9]
y_true = []
# Success ratio
success_ratio_in_xgb = []
success_ratio_oos_xgb = []

# Hyperparamètres spécifiques
best_params_xgb = []
mse_val_grids_xgb = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    if split_idx not in selected_splits:
        continue  
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_xgb):
        xgb_model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        xgb_model.fit(x_train[covariates], y_train)
        y_val_pred = xgb_model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_xgb.append(mse_grid)
    best_params_xgb.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    xgb_final = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    xgb_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = xgb_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_xgb.append(r2_in)

    # R² oos
    y_test_pred = xgb_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_xgb.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)
    y_true.append(y_test)

    # Stockage pour global
    y_pred_xgb.append(y_test_pred)
    
    y_trainval_xgb.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats

y_pred_xgb = np.concatenate(y_pred_xgb)
y_trainval_xgb = np.concatenate(y_trainval_xgb)
y_true = np.concatenate(y_true)

# Affichages
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_xgb):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_xgb):.6f}")


r2_oos = r2(y_true, y_pred_xgb)
print(r2_oos)

  0%|          | 0/10 [00:00<?, ?it/s]


Split 3 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 300, 'reg_alpha': 0, 'reg_lambda': 2} (MSE val = 0.004174)


 30%|███       | 3/10 [00:21<00:50,  7.29s/it]

R² in-sample : 0.038183 | R² oos : 0.122684

Split 5 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 300, 'reg_alpha': 0.5, 'reg_lambda': 2} (MSE val = 0.002791)


 50%|█████     | 5/10 [00:43<00:44,  8.99s/it]

R² in-sample : 0.036268 | R² oos : -0.023969

Split 7 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'reg_alpha': 0.5, 'reg_lambda': 2} (MSE val = 0.004292)


 70%|███████   | 7/10 [01:06<00:29,  9.98s/it]

R² in-sample : 0.058634 | R² oos : 0.110691

Split 9 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 300, 'reg_alpha': 0.5, 'reg_lambda': 0.5} (MSE val = 0.005204)


100%|██████████| 10/10 [01:29<00:00,  8.96s/it]

R² in-sample : 0.097465 | R² oos : 0.036646
Moyenne des R² in-sample (splits) : 0.057638
Moyenne des R² oos (splits) : 0.061513
0.05545532252752494
